In [ ]:
%load_ext autoreload
%autoreload 2

# Year-to-date US stock aggregates from Massive flat files

Download the `day_aggs` flat files for the current year-to-date range with
`MassiveFlatFiles`, then use them: per-ticker YTD returns and the market's top
movers.

Flat-files credentials are read from the environment (issued in the Massive
Dashboard, separate from `MASSIVE_API_KEY`):

```bash
export MASSIVE_S3_ACCESS_KEY=...
export MASSIVE_S3_SECRET_KEY=...
```

Note: downloaded files are reused as an on-disk cache (historical flat files
are immutable), so re-running is fast; pass `overwrite=True` to force a fresh
fetch of a recent day that may still be finalizing.

In [ ]:
import os
from datetime import date

import polars as pl

from xpectral.data import MassiveFlatFiles
from xpectral.utils import setup_logging

assert os.getenv("MASSIVE_S3_ACCESS_KEY") and os.getenv("MASSIVE_S3_SECRET_KEY"), (
    "set MASSIVE_S3_ACCESS_KEY and MASSIVE_S3_SECRET_KEY first"
)

# Turn on console logging so MassiveFlatFiles reports skipped/downloaded keys.
# force_color renders ANSI in the notebook (stderr here is not a TTY).
setup_logging("INFO", force_color=True)

pl.Config.set_tbl_rows(12)

## Download year-to-date

`day_aggs` is one row per ticker per day for the whole US market -- the smallest,
most practical dataset for a full-year pull. Missing days (weekends/holidays) are
skipped automatically.

In [ ]:
massive_ff = MassiveFlatFiles()

start = date(date.today().year, 1, 1)
end = date.today()
print(f"fetching day_aggs {start}..{end} ...")

df = massive_ff.get_flat_files(
    dataset="day_aggs",
    from_=start,
    to=end,
    prefix="us_stocks_sip",
    overwrite=False
).collect()

df.shape

In [3]:
print("rows:   ", df.height)
print("tickers:", df.get_column("ticker").n_unique())
print(
    "dates:  ",
    df.get_column("window_start").dt.date().min(),
    "->",
    df.get_column("window_start").dt.date().max(),
)
df.head()

rows:    1557821
tickers: 13704
dates:   2026-01-02 -> 2026-07-10


window_start,ticker,volume,open,close,high,low,transactions
"datetime[ns, America/New_York]",str,f64,f64,f64,f64,f64,i64
2026-01-02 00:00:00 EST,"""A""",1.650714e6,136.24,137.95,137.95,135.27,26678
2026-01-02 00:00:00 EST,"""AA""",5.897475e6,54.12,56.54,56.61,54.01,71955
2026-01-02 00:00:00 EST,"""AAA""",6358.0,24.98,24.95,25.0,24.95,71
2026-01-02 00:00:00 EST,"""AAAA""",1924.0,27.55,27.428,27.55,27.37,35
2026-01-02 00:00:00 EST,"""AAAC""",19.0,20.065,20.065,20.065,20.065,2


## A few tickers

Pivot closing prices for a handful of names into a wide, date-indexed table.

In [7]:
TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]

closes = (
    df.filter(pl.col("ticker").is_in(TICKERS))
    .select("window_start", "ticker", "close")
    .pivot(index="window_start", on="ticker", values="close")
    .sort("window_start")
)
closes.tail()

window_start,AAPL,AMZN,GOOGL,MSFT,NVDA
"datetime[ns, America/New_York]",f64,f64,f64,f64,f64
2026-07-06 00:00:00 EDT,312.66,244.16,366.46,386.74,195.55
2026-07-07 00:00:00 EDT,310.66,245.98,367.03,388.84,196.93
2026-07-08 00:00:00 EDT,313.39,243.62,361.92,383.34,204.12
2026-07-09 00:00:00 EDT,316.22,247.04,358.89,384.36,202.78
2026-07-10 00:00:00 EDT,315.32,245.34,357.18,385.1,210.96


## YTD returns

First close of the year vs the latest close, per ticker.

In [8]:
def ytd_return(frame: pl.DataFrame, tickers: list[str] | None = None) -> pl.DataFrame:
    if tickers is not None:
        frame = frame.filter(pl.col("ticker").is_in(tickers))
    return frame.group_by("ticker").agg(
        first_close=pl.col("close").sort_by("window_start").first(),
        last_close=pl.col("close").sort_by("window_start").last(),
    ).with_columns(ytd_return=pl.col("last_close") / pl.col("first_close") - 1)


ytd_return(df, TICKERS).sort("ytd_return", descending=True)

ticker,first_close,last_close,ytd_return
str,f64,f64,f64
"""AAPL""",271.01,315.32,0.1635
"""GOOGL""",315.15,357.18,0.133365
"""NVDA""",188.85,210.96,0.117077
"""AMZN""",226.5,245.34,0.083179
"""MSFT""",472.94,385.1,-0.185732


## Market-wide top movers

Because we pulled the whole market, we can rank YTD performance across every
ticker. Drop very low-priced names to avoid penny-stock noise.

In [9]:
movers = (
    ytd_return(df)
    .filter(pl.col("first_close") >= 5)  # drop sub-$5 noise
    .sort("ytd_return", descending=True)
)

print("Top 10 YTD gainers")
print(movers.head(10))
print("\nTop 10 YTD losers")
print(movers.tail(10).reverse())

Top 10 YTD gainers
shape: (10, 4)
┌────────┬─────────────┬────────────┬────────────┐
│ ticker ┆ first_close ┆ last_close ┆ ytd_return │
│ ---    ┆ ---         ┆ ---        ┆ ---        │
│ str    ┆ f64         ┆ f64        ┆ f64        │
╞════════╪═════════════╪════════════╪════════════╡
│ TCGL   ┆ 5.31        ┆ 172.84     ┆ 31.549906  │
│ BNY    ┆ 10.14       ┆ 151.92     ┆ 13.982249  │
│ MGRT   ┆ 7.2         ┆ 84.6       ┆ 10.75      │
│ OSCX   ┆ 9.9406      ┆ 114.9747   ┆ 10.566173  │
│ BWET   ┆ 18.7301     ┆ 203.35     ┆ 9.856856   │
│ UVIX   ┆ 5.47        ┆ 56.41      ┆ 9.312614   │
│ MRAL   ┆ 5.51        ┆ 56.39      ┆ 9.23412    │
│ TSLS   ┆ 5.22        ┆ 52.1686    ┆ 8.993985   │
│ LABX   ┆ 17.63       ┆ 154.0      ┆ 7.735111   │
│ DUST   ┆ 7.47        ┆ 62.7       ┆ 7.393574   │
└────────┴─────────────┴────────────┴────────────┘

Top 10 YTD losers
shape: (10, 4)
┌────────┬─────────────┬────────────┬────────────┐
│ ticker ┆ first_close ┆ last_close ┆ ytd_return │
│ ---    ┆ ---